In [66]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Load dataset
file_path = "cisco.csv"
df = pd.read_csv(file_path)

# Display dataset
df.head()


,Cost_Rank,Product_Name,Product_Life_Cycle,FY22_Q2,FY22_Q3,FY22_Q4,FY23_Q1,FY23_Q2,FY23_Q3,FY23_Q4,...,M_FY2025_Q1_BIAS,S_FY2024_Q3_Accuracy,S_FY2024_Q3_BIAS,S_FY2024_Q4_Accuracy,S_FY2024_Q4_BIAS,S_FY2025_Q1_Accuracy,S_FY2025_Q1_BIAS,D_FY2025_Q2_Forecasted,M_FY2025_Q2_Forecasted,S_FY2025_Q2_Forecasted
0,1,SWITCH Enterprise High,Sustaining,57147.0,52873.0,52870.0,38833.0,27114.0,21823,31813,...,70.11%,76.51%,23.49%,94.26%,-5.74%,84.94%,15.06%,29814,28378,26648
1,2,SWITCH Enterprise Ultra High,Sustaining,222.0,1549.0,4619.0,4764.0,5015.0,6656,9605,...,9.46%,93.05%,6.95%,46.30%,-53.70%,66.61%,-33.39%,11046,7748,11865
2,3,SWITCH Enterprise Ultra High,Sustaining,24362.0,21308.0,19067.0,14551.0,13271.0,10165,10477,...,81.29%,93.79%,-6.21%,8.94%,-91.06%,77.93%,-22.07%,10450,10785,9165
3,4,SWITCH Enterprise Low,Sustaining,NaN,NaN,1227.0,24186.0,7680.0,16772,17554,...,37.29%,78.81%,-21.19%,72.99%,-27.01%,48.12%,-51.88%,24505,29567,24186
4,5,TRANSCEIVER MODULE Mid,Sustaining,208760.0,116126.0,150803.0,82163.0,82408.0,67132,87498,...,42.22%,99.28%,-0.72%,89.31%,-10.69%,61.25%,38.75%,70000,71000,68858


In [68]:
# Convert percentage strings to numerical values for accuracy and bias columns
percentage_columns = [col for col in df.columns if "Accuracy" in col or "BIAS" in col]

# Remove '%' and convert to float
for col in percentage_columns:
    df[col] = df[col].str.rstrip('%').astype(float) / 100

# Check cleaned data
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 36 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Cost_Rank               20 non-null     int64  
 1   Product_Name            20 non-null     object 
 2   Product_Life_Cycle      20 non-null     object 
 3   FY22_Q2                 14 non-null     float64
 4   FY22_Q3                 14 non-null     float64
 5   FY22_Q4                 17 non-null     float64
 6   FY23_Q1                 17 non-null     float64
 7   FY23_Q2                 18 non-null     float64
 8   FY23_Q3                 20 non-null     int64  
 9   FY23_Q4                 20 non-null     int64  
 10  FY24_Q1                 20 non-null     int64  
 11  FY24_Q2                 20 non-null     int64  
 12  FY24_Q3                 20 non-null     int64  
 13  FY24_Q4                 20 non-null     int64  
 14  FY25_Q1                 20 non-null     int6

,Cost_Rank,Product_Name,Product_Life_Cycle,FY22_Q2,FY22_Q3,FY22_Q4,FY23_Q1,FY23_Q2,FY23_Q3,FY23_Q4,...,M_FY2025_Q1_BIAS,S_FY2024_Q3_Accuracy,S_FY2024_Q3_BIAS,S_FY2024_Q4_Accuracy,S_FY2024_Q4_BIAS,S_FY2025_Q1_Accuracy,S_FY2025_Q1_BIAS,D_FY2025_Q2_Forecasted,M_FY2025_Q2_Forecasted,S_FY2025_Q2_Forecasted
0,1,SWITCH Enterprise High,Sustaining,57147.0,52873.0,52870.0,38833.0,27114.0,21823,31813,...,0.7011,0.7651,0.2349,0.9426,-0.0574,0.8494,0.1506,29814,28378,26648
1,2,SWITCH Enterprise Ultra High,Sustaining,222.0,1549.0,4619.0,4764.0,5015.0,6656,9605,...,0.0946,0.9305,0.0695,0.4630,-0.5370,0.6661,-0.3339,11046,7748,11865
2,3,SWITCH Enterprise Ultra High,Sustaining,24362.0,21308.0,19067.0,14551.0,13271.0,10165,10477,...,0.8129,0.9379,-0.0621,0.0894,-0.9106,0.7793,-0.2207,10450,10785,9165
3,4,SWITCH Enterprise Low,Sustaining,NaN,NaN,1227.0,24186.0,7680.0,16772,17554,...,0.3729,0.7881,-0.2119,0.7299,-0.2701,0.4812,-0.5188,24505,29567,24186
4,5,TRANSCEIVER MODULE Mid,Sustaining,208760.0,116126.0,150803.0,82163.0,82408.0,67132,87498,...,0.4222,0.9928,-0.0072,0.8931,-0.1069,0.6125,0.3875,70000,71000,68858


In [3]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.9/253.9 MB 6.1 MB/s eta 0:00:0000:0100:02


In [48]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from tqdm import tqdm  # Progress bar

# Load dataset
# file_path = "cisco.csv"  # Update with your file path
df = pd.read_csv(file_path)

# Display dataset
df.head()


,Cost_Rank,Product_Name,Product_Life_Cycle,FY22_Q2,FY22_Q3,FY22_Q4,FY23_Q1,FY23_Q2,FY23_Q3,FY23_Q4,...,M_FY2025_Q1_BIAS,S_FY2024_Q3_Accuracy,S_FY2024_Q3_BIAS,S_FY2024_Q4_Accuracy,S_FY2024_Q4_BIAS,S_FY2025_Q1_Accuracy,S_FY2025_Q1_BIAS,D_FY2025_Q2_Forecasted,M_FY2025_Q2_Forecasted,S_FY2025_Q2_Forecasted
0,1,SWITCH Enterprise High,Sustaining,57147.0,52873.0,52870.0,38833.0,27114.0,21823,31813,...,70.11%,76.51%,23.49%,94.26%,-5.74%,84.94%,15.06%,29814,28378,26648
1,2,SWITCH Enterprise Ultra High,Sustaining,222.0,1549.0,4619.0,4764.0,5015.0,6656,9605,...,9.46%,93.05%,6.95%,46.30%,-53.70%,66.61%,-33.39%,11046,7748,11865
2,3,SWITCH Enterprise Ultra High,Sustaining,24362.0,21308.0,19067.0,14551.0,13271.0,10165,10477,...,81.29%,93.79%,-6.21%,8.94%,-91.06%,77.93%,-22.07%,10450,10785,9165
3,4,SWITCH Enterprise Low,Sustaining,NaN,NaN,1227.0,24186.0,7680.0,16772,17554,...,37.29%,78.81%,-21.19%,72.99%,-27.01%,48.12%,-51.88%,24505,29567,24186
4,5,TRANSCEIVER MODULE Mid,Sustaining,208760.0,116126.0,150803.0,82163.0,82408.0,67132,87498,...,42.22%,99.28%,-0.72%,89.31%,-10.69%,61.25%,38.75%,70000,71000,68858


In [72]:
def preprocess_data(df):
    """
    Converts percentage strings to numerical values for accuracy and bias columns.
    """
    # Identify accuracy and bias columns
    percentage_columns = [col for col in df.columns if "Accuracy" in col or "BIAS" in col]
    
    # Convert percentage strings to float values
    for col in percentage_columns:
        df[col] = df[col].astype(float)  # If stored as numbers already

    return df

# Preprocess the data
df = preprocess_data(df)

# Check cleaned data
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 36 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Cost_Rank               20 non-null     int64  
 1   Product_Name            20 non-null     object 
 2   Product_Life_Cycle      20 non-null     object 
 3   FY22_Q2                 14 non-null     float64
 4   FY22_Q3                 14 non-null     float64
 5   FY22_Q4                 17 non-null     float64
 6   FY23_Q1                 17 non-null     float64
 7   FY23_Q2                 18 non-null     float64
 8   FY23_Q3                 20 non-null     int64  
 9   FY23_Q4                 20 non-null     int64  
 10  FY24_Q1                 20 non-null     int64  
 11  FY24_Q2                 20 non-null     int64  
 12  FY24_Q3                 20 non-null     int64  
 13  FY24_Q4                 20 non-null     int64  
 14  FY25_Q1                 20 non-null     int6

,Cost_Rank,Product_Name,Product_Life_Cycle,FY22_Q2,FY22_Q3,FY22_Q4,FY23_Q1,FY23_Q2,FY23_Q3,FY23_Q4,...,M_FY2025_Q1_BIAS,S_FY2024_Q3_Accuracy,S_FY2024_Q3_BIAS,S_FY2024_Q4_Accuracy,S_FY2024_Q4_BIAS,S_FY2025_Q1_Accuracy,S_FY2025_Q1_BIAS,D_FY2025_Q2_Forecasted,M_FY2025_Q2_Forecasted,S_FY2025_Q2_Forecasted
0,1,SWITCH Enterprise High,Sustaining,57147.0,52873.0,52870.0,38833.0,27114.0,21823,31813,...,0.7011,0.7651,0.2349,0.9426,-0.0574,0.8494,0.1506,29814,28378,26648
1,2,SWITCH Enterprise Ultra High,Sustaining,222.0,1549.0,4619.0,4764.0,5015.0,6656,9605,...,0.0946,0.9305,0.0695,0.4630,-0.5370,0.6661,-0.3339,11046,7748,11865
2,3,SWITCH Enterprise Ultra High,Sustaining,24362.0,21308.0,19067.0,14551.0,13271.0,10165,10477,...,0.8129,0.9379,-0.0621,0.0894,-0.9106,0.7793,-0.2207,10450,10785,9165
3,4,SWITCH Enterprise Low,Sustaining,NaN,NaN,1227.0,24186.0,7680.0,16772,17554,...,0.3729,0.7881,-0.2119,0.7299,-0.2701,0.4812,-0.5188,24505,29567,24186
4,5,TRANSCEIVER MODULE Mid,Sustaining,208760.0,116126.0,150803.0,82163.0,82408.0,67132,87498,...,0.4222,0.9928,-0.0072,0.8931,-0.1069,0.6125,0.3875,70000,71000,68858


In [74]:
# Function to compute forecasted values based on actual sales and bias
def compute_forecasted_values(actual, bias):
    return actual * (1 + bias)

# Define the exact column names from your dataset
actual_quarters = ['FY24_Q3', 'FY24_Q4', 'FY25_Q1']
bias_quarters = ['FY2024_Q3', 'FY2024_Q4', 'FY2025_Q1']  # Adjust if needed

teams = ['D', 'M', 'S']

# Compute forecasted values for each team and each quarter
for team in teams:
    for i in range(len(actual_quarters)):
        actual_col = actual_quarters[i]  # Actual sales column
        bias_col = f"{team}_{bias_quarters[i]}_BIAS"  # Bias column for each team
        forecast_col = f"{team}_{actual_quarters[i]}_Forecasted"  # New column to store forecasted values

        # Check if both actual sales and bias columns exist before calculation
        if actual_col in df.columns and bias_col in df.columns:
            df[forecast_col] = compute_forecasted_values(df[actual_col], df[bias_col])
        else:
            print(f"Skipping {forecast_col} because {actual_col} or {bias_col} is missing.")

# Display the first few rows to verify forecasted values
df[[col for col in df.columns if 'Forecasted' in col]].head()


,D_FY2025_Q2_Forecasted,M_FY2025_Q2_Forecasted,S_FY2025_Q2_Forecasted,D_FY24_Q3_Forecasted,D_FY24_Q4_Forecasted,D_FY25_Q1_Forecasted,M_FY24_Q3_Forecasted,M_FY24_Q4_Forecasted,M_FY25_Q1_Forecasted,S_FY24_Q3_Forecasted,S_FY24_Q4_Forecasted,S_FY25_Q1_Forecasted
0,29814,28378,26648,25999.5300,29853.8812,29553.9972,27278.7680,38598.6308,41707.5698,26957.8670,27716.2104,28210.4108
1,11046,7748,11865,9823.5296,11605.5792,10979.4400,6919.5624,9790.5600,10579.3090,9180.5880,5811.5760,6437.8565
2,10450,10785,9165,10199.7972,10911.7236,10234.7082,10895.3220,15416.9184,16658.7381,8674.6371,960.3348,7160.9877
3,24505,29567,24186,25008.7012,26605.4800,22000.4352,25315.3474,39039.6940,44987.1872,19179.9897,16049.0412,15767.9616
4,70000,71000,68858,69002.3200,81098.6974,75000.0922,78999.2040,71001.9352,77501.3668,76286.7520,76289.4951,75610.4250


In [58]:
# Define historical actual sales columns (12 past quarters)
historical_quarters = ["FY22_Q2", "FY22_Q3", "FY22_Q4", "FY23_Q1", "FY23_Q2", "FY23_Q3", 
                       "FY23_Q4", "FY24_Q1", "FY24_Q2", "FY24_Q3", "FY24_Q4", "FY25_Q1"]

# Accuracy & Bias columns
accuracy_bias_columns = [col for col in df.columns if "Accuracy" in col or "BIAS" in col]

# Forecast values for last 3 quarters
forecast_columns = ["D_FY24_Q3_Forecasted", "M_FY24_Q3_Forecasted", "S_FY24_Q3_Forecasted",
                    "D_FY24_Q4_Forecasted", "M_FY24_Q4_Forecasted", "S_FY24_Q4_Forecasted",
                    "D_FY25_Q1_Forecasted", "M_FY25_Q1_Forecasted", "S_FY25_Q1_Forecasted"]

# Forecast values for FY25_Q2 (ONLY used for prediction)
FY25_Q2_forecasts = ["D_FY2025_Q2_Forecasted", "M_FY2025_Q2_Forecasted", "S_FY2025_Q2_Forecasted"]

# Features for training
X = df[historical_quarters + accuracy_bias_columns + forecast_columns]

# Targets (Actual sales for last 3 quarters)
target_columns = ["FY24_Q3", "FY24_Q4", "FY25_Q1"]

# Display feature set
X.head()


,FY22_Q2,FY22_Q3,FY22_Q4,FY23_Q1,FY23_Q2,FY23_Q3,FY23_Q4,FY24_Q1,FY24_Q2,FY24_Q3,...,S_FY2025_Q1_BIAS,D_FY24_Q3_Forecasted,M_FY24_Q3_Forecasted,S_FY24_Q3_Forecasted,D_FY24_Q4_Forecasted,M_FY24_Q4_Forecasted,S_FY24_Q4_Forecasted,D_FY25_Q1_Forecasted,M_FY25_Q1_Forecasted,S_FY25_Q1_Forecasted
0,57147.0,52873.0,52870.0,38833.0,27114.0,21823,31813,27302,22084,21830,...,0.1506,25999.5300,27278.7680,26957.8670,29853.8812,38598.6308,27716.2104,29553.9972,41707.5698,28210.4108
1,222.0,1549.0,4619.0,4764.0,5015.0,6656,9605,5956,8450,8584,...,-0.3339,9823.5296,6919.5624,9180.5880,11605.5792,9790.5600,5811.5760,10979.4400,10579.3090,6437.8565
2,24362.0,21308.0,19067.0,14551.0,13271.0,10165,10477,7295,8084,9249,...,-0.2207,10199.7972,10895.3220,8674.6371,10911.7236,15416.9184,960.3348,10234.7082,16658.7381,7160.9877
3,NaN,NaN,1227.0,24186.0,7680.0,16772,17554,16095,26125,24337,...,-0.5188,25008.7012,25315.3474,19179.9897,26605.4800,39039.6940,16049.0412,22000.4352,44987.1872,15767.9616
4,208760.0,116126.0,150803.0,82163.0,82408.0,67132,87498,69599,78130,76840,...,0.3875,69002.3200,78999.2040,76286.7520,81098.6974,71001.9352,76289.4951,75000.0922,77501.3668,75610.4250


In [60]:
# Initialize dictionary to store models & MAE scores
models = {}
mae_scores = {}

# Train models for each quarter we are predicting
for target in tqdm(target_columns, desc="Training Models"):
    y_single = df[target]  # Target column (actual sales)
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y_single, test_size=0.2, random_state=42)
    
    # Initialize and train XGBoost model
    model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
    model.fit(X_train, y_train)
    
    # Predict and evaluate
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    
    # Store results
    models[target] = model
    mae_scores[target] = mae

# Display MAE scores
print("Model Performance (Lower MAE is better):")
for target, score in mae_scores.items():
    print(f"{target}: MAE = {score:.4f}")


Training Models: 100%|██████████| 3/3 [00:00<00:00, 20.71it/s]

Model Performance (Lower MAE is better):
FY24_Q3: MAE = 3195.6335
FY24_Q4: MAE = 4578.9640
FY25_Q1: MAE = 6430.4141


In [62]:
# Use the trained model for FY25_Q1 to predict FY25_Q2 (since it's the closest quarter)
model_for_FY25_Q2 = models["FY25_Q1"]

# Prepare input features for FY25_Q2
X_FY25_Q2 = df[historical_quarters + accuracy_bias_columns + FY25_Q2_forecasts]

# Predict FY25_Q2 actual sales
df["FY25_Q2_Predicted"] = model_for_FY25_Q2.predict(X_FY25_Q2)

# Show predictions
print("\nFY25_Q2 Predicted Sales:")
print(df[["FY25_Q2_Predicted"]].head())


ValueError: feature_names mismatch: ['FY22_Q2', 'FY22_Q3', 'FY22_Q4', 'FY23_Q1', 'FY23_Q2', 'FY23_Q3', 'FY23_Q4', 'FY24_Q1', 'FY24_Q2', 'FY24_Q3', 'FY24_Q4', 'FY25_Q1', 'D_FY2024_Q3_Accuracy', 'D_FY2024_Q3_BIAS', 'D_FY2024_Q4_Accuracy', 'D_FY2024_Q4_BIAS', 'D_FY2025_Q1_Accuracy', 'D_FY2025_Q1_BIAS', 'M_FY2024_Q3_Accuracy', 'M_FY2024_Q3_BIAS', 'M_FY2024_Q4_Accuracy', 'M_FY2024_Q4_BIAS', 'M_FY2025_Q1_Accuracy', 'M_FY2025_Q1_BIAS', 'S_FY2024_Q3_Accuracy', 'S_FY2024_Q3_BIAS', 'S_FY2024_Q4_Accuracy', 'S_FY2024_Q4_BIAS', 'S_FY2025_Q1_Accuracy', 'S_FY2025_Q1_BIAS', 'D_FY24_Q3_Forecasted', 'M_FY24_Q3_Forecasted', 'S_FY24_Q3_Forecasted', 'D_FY24_Q4_Forecasted', 'M_FY24_Q4_Forecasted', 'S_FY24_Q4_Forecasted', 'D_FY25_Q1_Forecasted', 'M_FY25_Q1_Forecasted', 'S_FY25_Q1_Forecasted'] ['FY22_Q2', 'FY22_Q3', 'FY22_Q4', 'FY23_Q1', 'FY23_Q2', 'FY23_Q3', 'FY23_Q4', 'FY24_Q1', 'FY24_Q2', 'FY24_Q3', 'FY24_Q4', 'FY25_Q1', 'D_FY2024_Q3_Accuracy', 'D_FY2024_Q3_BIAS', 'D_FY2024_Q4_Accuracy', 'D_FY2024_Q4_BIAS'

In [76]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from tqdm import tqdm  # Progress bar for training

# Load dataset
# file_path = "cisco.csv"  # Update with correct path if needed
df = pd.read_csv(file_path)

def preprocess_data(df):
    """
    Converts percentage strings to numerical values for accuracy and bias columns.
    """
    # Identify accuracy and bias columns
    percentage_columns = [col for col in df.columns if "Accuracy" in col or "BIAS" in col]
    
    # Convert percentage strings to float values
    for col in percentage_columns:
        df[col] = df[col].str.replace('%', '').astype(float) / 100
    
    return df

# Preprocess the data
df = preprocess_data(df)

# Define feature columns (X) and target variables (y)
historical_quarters = ["FY22_Q2", "FY22_Q3", "FY22_Q4", "FY23_Q1", "FY23_Q2", "FY23_Q3", 
                       "FY23_Q4", "FY24_Q1", "FY24_Q2", "FY24_Q3", "FY24_Q4", "FY25_Q1"]
accuracy_bias_columns = [col for col in df.columns if "Accuracy" in col or "BIAS" in col]
forecast_columns = ["D_FY24_Q3_Forecasted", "M_FY24_Q3_Forecasted", "S_FY24_Q3_Forecasted", 
                    "D_FY24_Q4_Forecasted", "M_FY24_Q4_Forecasted", "S_FY24_Q4_Forecasted", 
                    "D_FY25_Q1_Forecasted", "M_FY25_Q1_Forecasted", "S_FY25_Q1_Forecasted"]
forecasted_FY25_Q2 = ["D_FY2025_Q2_Forecasted", "M_FY2025_Q2_Forecasted", "S_FY2025_Q2_Forecasted"]

# Feature set for training (excluding FY25_Q2 forecasts)
X_train = df[historical_quarters + accuracy_bias_columns + forecast_columns]

# Target variables
target_columns = ["FY24_Q3", "FY24_Q4", "FY25_Q1"]

# Initialize dictionary to store models and MAE scores
models = {}
mae_scores = {}

# Train models for each target variable
for target in tqdm(target_columns, desc="Training Models"):
    y_single = df[target]  # Target column
    
    # Train/test split
    X_train_split, X_test_split, y_train, y_test = train_test_split(X_train, y_single, test_size=0.2, random_state=42)
    
    # Initialize and train XGBoost model
    model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
    model.fit(X_train_split, y_train)
    
    # Predict and evaluate
    y_pred = model.predict(X_test_split)
    mae = mean_absolute_error(y_test, y_pred)
    
    # Store results
    models[target] = model
    mae_scores[target] = mae

# Display MAE scores
print("Model Performance (Lower MAE is better):")
for target, score in mae_scores.items():
    print(f"{target}: MAE = {score:.4f}")

# Prepare features for FY25_Q2 prediction (use FY25_Q2 forecasts now)
X_FY25_Q2 = df[historical_quarters + accuracy_bias_columns + forecasted_FY25_Q2]

# Predict FY25_Q2 using trained models
predictions_FY25_Q2 = {}
for target, model in models.items():
    predictions_FY25_Q2[target] = model.predict(X_FY25_Q2)  # Using FY25_Q2 forecast data

# Convert predictions to DataFrame
predictions_df = pd.DataFrame(predictions_FY25_Q2)
print("\nFY25_Q2 Predictions:")
print(predictions_df.head())


KeyError: "['D_FY24_Q3_Forecasted', 'M_FY24_Q3_Forecasted', 'S_FY24_Q3_Forecasted', 'D_FY24_Q4_Forecasted', 'M_FY24_Q4_Forecasted', 'S_FY24_Q4_Forecasted', 'D_FY25_Q1_Forecasted', 'M_FY25_Q1_Forecasted', 'S_FY25_Q1_Forecasted'] not in index"

In [78]:
print("Available columns in dataset:", df.columns.tolist())


Available columns in dataset: ['Cost_Rank', 'Product_Name', 'Product_Life_Cycle', 'FY22_Q2', 'FY22_Q3', 'FY22_Q4', 'FY23_Q1', 'FY23_Q2', 'FY23_Q3', 'FY23_Q4', 'FY24_Q1', 'FY24_Q2', 'FY24_Q3', 'FY24_Q4', 'FY25_Q1', 'D_FY2024_Q3_Accuracy', 'D_FY2024_Q3_BIAS', 'D_FY2024_Q4_Accuracy', 'D_FY2024_Q4_BIAS', 'D_FY2025_Q1_Accuracy', 'D_FY2025_Q1_BIAS', 'M_FY2024_Q3_Accuracy', 'M_FY2024_Q3_BIAS', 'M_FY2024_Q4_Accuracy', 'M_FY2024_Q4_BIAS', 'M_FY2025_Q1_Accuracy', 'M_FY2025_Q1_BIAS', 'S_FY2024_Q3_Accuracy', 'S_FY2024_Q3_BIAS', 'S_FY2024_Q4_Accuracy', 'S_FY2024_Q4_BIAS', 'S_FY2025_Q1_Accuracy', 'S_FY2025_Q1_BIAS', 'D_FY2025_Q2_Forecasted', 'M_FY2025_Q2_Forecasted', 'S_FY2025_Q2_Forecasted']


In [82]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from tqdm import tqdm  # Progress bar for training

# Load dataset
# file_path = "cisco.csv"  # Update with correct path if needed
df = pd.read_csv(file_path)

def preprocess_data(df):
    """
    Converts percentage strings to numerical values for accuracy and bias columns.
    """
    # Identify accuracy and bias columns
    percentage_columns = [col for col in df.columns if "Accuracy" in col or "BIAS" in col]
    
    # Convert percentage strings to float values
    for col in percentage_columns:
        df[col] = df[col].str.replace('%', '').astype(float) / 100
    
    return df

# Preprocess the data
df = preprocess_data(df)

# Define feature columns (X) and target variables (y)
historical_quarters = ["FY22_Q2", "FY22_Q3", "FY22_Q4", "FY23_Q1", "FY23_Q2", "FY23_Q3", 
                       "FY23_Q4", "FY24_Q1", "FY24_Q2", "FY24_Q3", "FY24_Q4", "FY25_Q1"]
accuracy_bias_columns = [col for col in df.columns if "Accuracy" in col or "BIAS" in col]
forecast_columns = [col for col in df.columns if "Forecasted" in col and "FY25_Q2" not in col]
forecasted_FY25_Q2 = [col for col in df.columns if "FY25_Q2_Forecasted" in col]

# Feature set for training (excluding FY25_Q2 forecasts)
X_train = df[historical_quarters + accuracy_bias_columns + forecast_columns]

# Target variables
target_columns = ["FY24_Q3", "FY24_Q4", "FY25_Q1"]

# Initialize dictionary to store models and MAE scores
models = {}
mae_scores = {}

# Train models for each target variable
for target in tqdm(target_columns, desc="Training Models"):
    y_single = df[target]  # Target column
    
    # Train/test split
    X_train_split, X_test_split, y_train, y_test = train_test_split(X_train, y_single, test_size=0.2, random_state=42)
    
    # Initialize and train XGBoost model
    model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
    model.fit(X_train_split, y_train)
    
    # Predict and evaluate
    y_pred = model.predict(X_test_split)
    mae = mean_absolute_error(y_test, y_pred)
    
    # Store results
    models[target] = model
    mae_scores[target] = mae

# Display MAE scores
print("Model Performance (Lower MAE is better):")
for target, score in mae_scores.items():
    print(f"{target}: MAE = {score:.4f}")

# Prepare features for FY25_Q2 prediction (use only columns from X_train)
X_FY25_Q2 = df[X_train.columns]

# Predict FY25_Q2 using trained models
predictions_FY25_Q2 = {}
for target, model in models.items():
    predictions_FY25_Q2[target] = model.predict(X_FY25_Q2)  # Using aligned features

# Convert predictions to DataFrame
predictions_df = pd.DataFrame(predictions_FY25_Q2)
print("\nFY25_Q2 Predictions:")
print(predictions_df.head())


Training Models: 100%|██████████| 3/3 [00:00<00:00, 19.89it/s]

Model Performance (Lower MAE is better):
FY24_Q3: MAE = 3195.6335
FY24_Q4: MAE = 4579.1877
FY25_Q1: MAE = 6430.4141

FY25_Q2 Predictions:
        FY24_Q3       FY24_Q4       FY25_Q1
0  29370.667969  21989.167969  16498.101562
1   7091.809082   6736.406250   4997.252930
2   9250.752930  10743.736328   9190.366211
3  24337.972656  21989.167969  32764.859375
4  76755.867188  85325.484375  54474.953125


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from tqdm import tqdm  # Progress bar for training

# Load dataset
file_path = "cisco.csv"  # Update with correct path if needed
df = pd.read_csv(file_path)

def preprocess_data(df):
    """
    Converts percentage strings to numerical values for accuracy and bias columns.
    """
    # Identify accuracy and bias columns
    percentage_columns = [col for col in df.columns if "Accuracy" in col or "BIAS" in col]
    
    # Convert percentage strings to float values
    for col in percentage_columns:
        df[col] = df[col].str.replace('%', '').astype(float) / 100
    
    return df

# Preprocess the data
df = preprocess_data(df)

# Define feature columns (X) and target variables (y)
historical_quarters = ["FY22_Q2", "FY22_Q3", "FY22_Q4", "FY23_Q1", "FY23_Q2", "FY23_Q3", 
                       "FY23_Q4", "FY24_Q1", "FY24_Q2", "FY24_Q3", "FY24_Q4", "FY25_Q1"]
accuracy_bias_columns = [col for col in df.columns if "Accuracy" in col or "BIAS" in col]
forecast_columns = [col for col in df.columns if "Forecasted" in col and "FY25_Q2" not in col]
forecasted_FY25_Q2 = [col for col in df.columns if "FY25_Q2_Forecasted" in col]

# Feature set for training (excluding FY25_Q2 forecasts)
X_train = df[historical_quarters + accuracy_bias_columns + forecast_columns]

# Target variables
target_columns = ["FY24_Q3", "FY24_Q4", "FY25_Q1", "FY25_Q2"]

# Initialize dictionary to store models and MAE scores
models = {}
mae_scores = {}

# Train models for each target variable
for target in tqdm(target_columns, desc="Training Models"):
    if target not in df.columns:
        continue  # Skip if FY25_Q2 actuals are missing
    
    y_single = df[target]  # Target column
    
    # Train/test split
    X_train_split, X_test_split, y_train, y_test = train_test_split(X_train, y_single, test_size=0.2, random_state=42)
    
    # Initialize and train XGBoost model
    model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
    model.fit(X_train_split, y_train)
    
    # Predict and evaluate
    y_pred = model.predict(X_test_split)
    mae = mean_absolute_error(y_test, y_pred)
    
    # Store results
    models[target] = model
    mae_scores[target] = mae

# Display MAE scores
print("Model Performance (Lower MAE is better):")
for target, score in mae_scores.items():
    print(f"{target}: MAE = {score:.4f}")

# Prepare features for FY25_Q2 prediction (use only columns from X_train)
X_FY25_Q2 = df[X_train.columns]

# Predict FY25_Q2 using trained model (if available)
if "FY25_Q2" in models:
    df["Predicted_FY25_Q2"] = models["FY25_Q2"].predict(X_FY25_Q2)
    print("\nFY25_Q2 Sales Predictions:")
    print(df[["Product_Name", "Predicted_FY25_Q2"]].head())
